# 📟 Stage 3 — On-Device Deployment & Measurement

**Project:** Energy-Aware Visual Anomaly Detection on MCUs — Stage 3 / 4

Takes the Stage 2 compressed models, converts them to TFLite INT8, deploys them on the **Arduino Nano 33 BLE Sense Rev 2**, and measures real latency, memory, and energy per inference.

## The two halves of Stage 3

**Half A — in Colab (this notebook):** PyTorch → ONNX → TFLite INT8 conversion, with numerical verification that the TFLite output matches PyTorch, and conversion of each model to a C header array for the Arduino.

**Half B — on the Arduino (C++ firmware, provided as text cells):** load a model, run inference on a stored test image, report inference latency and memory. Energy is measured externally with a current shunt / power profiler.

## What replaces what

The **simulated INT8** numbers from Stage 2 (theoretical size, weight-only fake-quant accuracy) are replaced here by **real measurements**: actual TFLite INT8 model size, measured on-device latency, measured SRAM/flash usage, and measured energy. The Stage 2 Pareto plot gets its size/energy axes updated with hardware truth.

## The conversion is the hard part

PyTorch → TFLite for conv-heavy models with transpose convolutions is historically fragile. We convert **one model end-to-end and verify it first**, then batch. If a model fails to convert, the notebook reports which one and why rather than silently producing a broken artifact.

---

# Zone 1 — Setup

## 1. Install conversion toolchain
Restart the runtime once if imports fail in the next cell (TF + ONNX sometimes need a fresh interpreter).

In [1]:
!pip install -q onnx onnxruntime onnx2tf tensorflow ai-edge-litert
!pip install -q onnx-graphsurgeon sng4onnx onnxsim
print('Toolchain installed.')

Toolchain installed.


In [2]:
!pip install -q onnxscript

In [6]:
#!pip install -q ai-edge-torch
# Clean install: ai-edge-torch with its required torch version, all at once
!pip install -q ai-edge-torch 2>/dev/null || pip install -q ai-edge-torch

In [1]:
!pip uninstall -y tensorflow tensorflow-cpu tf-nightly tf-keras keras ai-edge-torch litert-torch
!pip install -U pip
!pip install ai-edge-torch

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
Found existing installation: tf_keras 2.20.0
Uninstalling tf_keras-2.20.0:
  Successfully uninstalled tf_keras-2.20.0
Found existing installation: keras 3.13.2
Uninstalling keras-3.13.2:
  Successfully uninstalled keras-3.13.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.8/575.8 kB 18.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 100.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 54.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 MB 51.7 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 130.6 MB/s  0:00:00
  Attempting uninstall: torchao
    Found exist

In [3]:
!pip install -U tensorflow==2.20.0 keras==3.10.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 36.8 MB/s  0:00:10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [tensorflow]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tf-keras>=2.18.0, which is not installed.
keras-hub 0.26.0 requires keras>=3.13, but you have keras 3.10.0 which is incompatible.


In [2]:
!pip list | grep -E "tensorflow|keras|ai-edge|litert|numpy|ml-dtypes"

ai-edge-litert                           2.1.5
ai-edge-quantizer                        0.7.0
ai-edge-torch                            0.7.2
keras-hub                                0.26.0
keras-nlp                                0.26.0
litert-converter                         0.2.0
litert-lm-builder                        0.13.0
litert-torch                             0.9.1
numpy                                    2.0.2
tensorflow-datasets                      4.9.10
tensorflow-hub                           0.16.1
tensorflow-metadata                      1.21.0
tensorflow-probability                   0.25.0
tensorflow-text                          2.20.1


In [10]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)

import ai_edge_torch
print("ai_edge_torch imported successfully!")

TensorFlow: 2.20.0
ai_edge_torch imported successfully!


In [9]:
import ai_edge_torch
print(ai_edge_torch.__version__)
print(dir(ai_edge_torch))

AttributeError: module 'ai_edge_torch' has no attribute '__version__'

In [15]:
pip show ai-edge-torch


Name: ai-edge-torch
Version: 0.7.2
Summary: DEPRECATED: renamed to litert-torch
Home-page: 
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: litert-torch
Required-by: 


In [16]:
pip install -U ai-edge-torch

In [19]:
!pip uninstall -y ai-edge-torch
!pip install -U litert-torch

Found existing installation: ai-edge-torch 0.7.2
Uninstalling ai-edge-torch-0.7.2:
  Successfully uninstalled ai-edge-torch-0.7.2


In [53]:
!pip install -q ai-edge-quantizer

## 2. Imports, Drive, config

In [2]:
import os, json, shutil
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

from google.colab import drive
drive.mount('/content/drive')

# symlink data
src = '/content/drive/MyDrive/mvtec'; dst = '/content/mvtec'
if os.path.islink(dst): os.unlink(dst)
elif os.path.isdir(dst): shutil.rmtree(dst)
os.symlink(src, dst)

# pull Stage 2 models from Drive
STAGE2_DIR = Path('/content/stage2_out'); STAGE2_DIR.mkdir(exist_ok=True)
drive_s2 = Path('/content/drive/MyDrive/mvtec_stage2')
for f in drive_s2.glob('*'):
    shutil.copy(f, STAGE2_DIR / f.name)

OUT = Path('/content/stage3_out'); OUT.mkdir(exist_ok=True)
(OUT / 'tflite').mkdir(exist_ok=True)
(OUT / 'headers').mkdir(exist_ok=True)

CFG = {'data_root': '/content/mvtec', 'image_size': 128,
       'base_channels': 32, 'latent_channels': 32, 'n_down': 3, 'batch_size': 32}
PROTOCOL = {'border': 8, 'blur_kernel': 5, 'pool': 'mean'}

print('Stage 2 models available:')
for f in sorted(STAGE2_DIR.glob('model_*.pt')): print('  ', f.name)

Mounted at /content/drive
Stage 2 models available:
   model_bottle_Baseline_FP32.pt
   model_bottle_Distill_b16.pt
   model_bottle_Distill_b16_plus_INT8.pt
   model_bottle_INT8_sim.pt
   model_bottle_Prune_30pct_l1.pt
   model_bottle_Prune_30pct_l1_plus_INT8.pt
   model_bottle_Prune_30pct_taylor.pt
   model_bottle_Prune_30pct_taylor_plus_INT8.pt
   model_bottle_Prune_50pct_l1.pt
   model_bottle_Prune_50pct_l1_plus_INT8.pt
   model_bottle_Prune_50pct_taylor.pt
   model_bottle_Prune_50pct_taylor_plus_INT8.pt
   model_hazelnut_Baseline_FP32.pt
   model_hazelnut_Distill_b16.pt
   model_hazelnut_Distill_b16_plus_INT8.pt
   model_hazelnut_INT8_sim.pt
   model_hazelnut_Prune_30pct_l1.pt
   model_hazelnut_Prune_30pct_l1_plus_INT8.pt
   model_hazelnut_Prune_30pct_taylor.pt
   model_hazelnut_Prune_30pct_taylor_plus_INT8.pt
   model_hazelnut_Prune_50pct_l1.pt
   model_hazelnut_Prune_50pct_l1_plus_INT8.pt
   model_hazelnut_Prune_50pct_taylor.pt
   model_hazelnut_Prune_50pct_taylor_plus_INT8.pt


## 3. Model + data (to rebuild and verify)

In [3]:
def conv_block(in_c, out_c, stride=2):
    return nn.Sequential(nn.Conv2d(in_c, out_c, 4, stride, 1, bias=True), nn.ReLU(inplace=True))
def deconv_block(in_c, out_c, last=False):
    layers = [nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=True)]
    if not last: layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)

class CompactAE(nn.Module):
    def __init__(self, base=32, latent=32, n_down=3):
        super().__init__()
        chans = [3] + [base*(2**i) for i in range(n_down)] + [latent]
        self.enc = nn.Sequential(*[conv_block(chans[i], chans[i+1]) for i in range(len(chans)-1)])
        rev = list(reversed(chans))
        self.dec = nn.Sequential(*[deconv_block(rev[i], rev[i+1], last=(i==len(rev)-2))
                                   for i in range(len(rev)-1)], nn.Sigmoid())
    def forward(self, x): return self.dec(self.enc(x))

def rebuild_from_checkpoint(ckpt_path):
    """Rebuild a (possibly pruned) CompactAE from its checkpoint, using the
    actual weight shapes in the state_dict so pruned architectures load correctly."""
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    sd = ckpt['state_dict']
    model = CompactAE(CFG['base_channels'], CFG['latent_channels'], CFG['n_down'])
    try:
        model.load_state_dict(sd); model.eval(); return model, ckpt
    except RuntimeError:
        pass  # pruned -> rebuild conv shapes from state_dict
    for i in range(len(model.enc)):
        w = sd[f'enc.{i}.0.weight']
        model.enc[i][0] = nn.Conv2d(w.shape[1], w.shape[0], 4, 2, 1, bias=True)
    for i in range(len(model.dec)-1):
        key = f'dec.{i}.0.weight'
        if key not in sd: continue
        w = sd[key]
        model.dec[i][0] = nn.ConvTranspose2d(w.shape[0], w.shape[1], 4, 2, 1, bias=True)
    model.load_state_dict(sd); model.eval()
    return model, ckpt

# Test dataset for calibration + verification
class MVTecTest(torch.utils.data.Dataset):
    def __init__(self, root, image_size):
        self.items = []
        for sub in sorted(Path(root, 'test').iterdir()):
            if not sub.is_dir(): continue
            label = 0 if sub.name == 'good' else 1
            for p in sorted(sub.rglob('*.png')): self.items.append((p, label, sub.name))
        self.tf = transforms.Compose([transforms.Resize(image_size+16),
                                      transforms.CenterCrop(image_size), transforms.ToTensor()])
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, l, d = self.items[i]
        return self.tf(Image.open(p).convert('RGB')), l, d

class MVTecTrain(torch.utils.data.Dataset):
    def __init__(self, root, image_size):
        self.paths = sorted(Path(root, 'train/good').rglob('*.png'))
        self.tf = transforms.Compose([transforms.Resize(image_size+16),
                                      transforms.CenterCrop(image_size), transforms.ToTensor()])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(self.paths[i]).convert('RGB'))

# Zone 2 — Conversion (Half A)
*PyTorch → ONNX → TFLite INT8. Prove on one model, then batch.*

## 4. The converter

Path: PyTorch → ONNX (opset 13) → `onnx2tf` → TFLite with full INT8 quantization. Full INT8 needs a **representative dataset** (real images) so the converter can calibrate activation ranges — we feed defect-free training images. Input and output are kept INT8 for the smallest, fastest model on the MCU.

In [5]:
import onnx2tf.utils.common_functions as cf
import numpy as np

# onnx2tf's download_test_image_data() fetches a calibration .npy that fails to load
# (allow_pickle mismatch). Replace it with a stub returning dummy data of the right shape.
# This only affects onnx2tf's internal cosmetic accuracy-check; our real verification
# is done separately in the TFLite-vs-PyTorch step.
def _stub_download():
    # shape onnx2tf expects: (N, H, W, C) float32 in [0,1]
    return np.random.rand(1, 224, 224, 3).astype(np.float32)

cf.download_test_image_data = _stub_download
print('Patched onnx2tf downloader.')

Patched onnx2tf downloader.


In [21]:
import litert_torch
import torch, numpy as np

def export_to_tflite(ckpt_path, name, rep_images, out_dir=OUT):
    model, ckpt = rebuild_from_checkpoint(ckpt_path)
    model.eval()

    # representative dataset for INT8 calibration: list of (input_tuple,)
    sample_inputs = tuple(
        img.unsqueeze(0)  # NCHW, float [0,1]
        for img in rep_images[:100]
    )

    # ai-edge-torch quantization recipe (full INT8)
    from ai_edge_torch.quantize import pt2e_quantizer, quant_config
    from torch.ao.quantization.quantize_pt2e import prepare_pt2e, convert_pt2e

    quantizer = pt2e_quantizer.PT2EQuantizer().set_global(
        pt2e_quantizer.get_symmetric_quantization_config(is_per_channel=True))
    sample = (rep_images[0].unsqueeze(0),)
    model_exported = torch.export.export_for_training(model, sample).module()
    model_prepared = prepare_pt2e(model_exported, quantizer)
    # calibrate
    with torch.no_grad():
        for img in rep_images[:100]:
            model_prepared(img.unsqueeze(0))
    model_quant = convert_pt2e(model_prepared)

    edge_model = ai_edge_torch.convert(
        model_quant, sample,
        quant_config=quant_config.QuantConfig(pt2e_quantizer=quantizer))

    tflite_path = out_dir / 'tflite' / f'{name}.tflite'
    edge_model.export(str(tflite_path))
    size_kb = tflite_path.stat().st_size / 1024
    return tflite_path, size_kb, ckpt

## 5. Convert ONE model end-to-end
Pick the trickiest case first (a pruned model with transpose convs). If this converts and verifies, the rest will.

In [22]:
import litert_torch

def export_to_tflite_fp32(ckpt_path, name, rep_images, out_dir=OUT):
    model, ckpt = rebuild_from_checkpoint(ckpt_path)
    model.eval()
    sample = (rep_images[0].unsqueeze(0),)          # (1, 3, 128, 128)
    edge_model = litert_torch.convert(model.eval(), sample)
    tflite_path = out_dir / 'tflite' / f'{name}.tflite'
    edge_model.export(str(tflite_path))
    return tflite_path, tflite_path.stat().st_size / 1024, ckpt

In [65]:
import numpy as np
from ai_edge_quantizer import quantizer

from ai_edge_quantizer import quantizer, qtyping

def export_int8(ckpt_path, name, rep_images, out_dir=OUT):
    # 1. FP32 conversion
    model, ckpt = rebuild_from_checkpoint(ckpt_path)
    model.eval()
    sample = (rep_images[0].unsqueeze(0),)

    fp32_path = out_dir / 'tflite' / f'{name}_fp32.tflite'
    litert_torch.convert(model.eval(), sample).export(str(fp32_path))

    # 2. Quantizer
    qt = quantizer.Quantizer(str(fp32_path))

    # Weight quantization config
    weight_cfg = qtyping.TensorQuantizationConfig(
        num_bits=8,
        symmetric=True,
        granularity=qtyping.QuantGranularity.CHANNELWISE,
    )

    op_cfg = qtyping.OpQuantizationConfig(
        weight_tensor_config=weight_cfg,
        compute_precision=qtyping.ComputePrecision.INTEGER,
        explicit_dequantize=False,
    )

    # Apply recipe to all supported ops
    for op in qtyping.TFLOperationName:
        try:
            qt.update_quantization_recipe(
                regex='.*',
                operation_name=op,
                op_config=op_cfg,
                algorithm_key='min_max_uniform_quantize',
            )
        except Exception as e:
            print(f"Skipping {op}: {e}")

    # Representative dataset
    from ai_edge_litert.interpreter import Interpreter
    interp = Interpreter(model_path=str(fp32_path))
    interp.allocate_tensors()
    in_name = interp.get_input_details()[0]['name']

    def rep_data():
        for img in rep_images[:100]:
            yield {in_name: img.unsqueeze(0).numpy().astype(np.float32)}

    calib = qt.calibrate(rep_data)
    result = qt.quantize(calib)

    int8_path = out_dir / 'tflite' / f'{name}.tflite'
    result.export_model(str(int8_path))

    return int8_path, int8_path.stat().st_size / 1024, ckpt

In [24]:
import litert_torch
print('version:', getattr(litert_torch, '__version__', 'unknown'))
print('file:', getattr(litert_torch, '__file__', 'unknown'))
print('attrs:', [a for a in dir(litert_torch) if not a.startswith('_')])

version: 0.9.1
file: /usr/local/lib/python3.12/dist-packages/litert_torch/__init__.py
attrs: ['Model', 'backend', 'config', 'convert', 'experimental_add_compilation_backend', 'fx_infra', 'generative', 'load', 'model', 'progress', 'quantize', 'signature', 'to_channel_last_io', 'version']


In [61]:
IMG = CFG['image_size']
# Pick ONE model, wire everything to it
TEST_NAME = 'model_bottle_Prune_50pct_l1'
CAT = 'bottle'

ckpt_file = STAGE2_DIR / f'{TEST_NAME}.pt'
test_ds = MVTecTest(Path(CFG['data_root'])/CAT, IMG)

# rebuild + score PyTorch
model, ckpt = rebuild_from_checkpoint(ckpt_file)
s_pt, l_pt = score_pytorch(model, test_ds)

# convert + score LiteRT (same checkpoint, same test set)
rep_ds = MVTecTrain(Path(CFG['data_root'])/CAT, IMG)
rep_images = [rep_ds[i] for i in range(min(100, len(rep_ds)))]
tflite_path, size_kb, _ = export_to_tflite_fp32(ckpt_file, TEST_NAME, rep_images)
s_tf, l_tf = score_litert(tflite_path, test_ds)

print(f'Model         : {TEST_NAME} ({CAT})')
print(f'Checkpoint    : {ckpt["auroc"]:.4f}')
print(f'PyTorch rebuilt: {roc_auc_score(l_pt, s_pt):.4f}')
print(f'LiteRT FP32   : {roc_auc_score(l_tf, s_tf):.4f}')
print(f'Score corr    : {np.corrcoef(s_pt, s_tf)[0,1]:.4f}')



(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:00) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:00)

...

(00:01) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:00)

(00:01) [ DONE] LiteRT-Torch Convert (+00:01)

(00:00) [START] Write Model to /content/stage3_out/tflite/model_bottle_Prune_50pct_l1.tflite

(00:00) [ DONE] Write Model to /content/stage3_out/tflite/model_bottle_Prune_50pct_l1.tflite (+00:00)

Model         : model_bottle_Prune_50pct_l1 (bottle)
Checkpoint    : 0.9040
PyTorch rebuilt: 0.9040
LiteRT FP32   : 0.9040
Score corr    : 1.0000


In [57]:
import inspect
from ai_edge_quantizer import quantizer

print(inspect.signature(
    quantizer.Quantizer.update_quantization_recipe
))

(self, regex: str, operation_name: ai_edge_quantizer.qtyping.TFLOperationName, op_config: Optional[ai_edge_quantizer.qtyping.OpQuantizationConfig] = None, algorithm_key: str = <AlgorithmName.MIN_MAX_UNIFORM_QUANT: 'min_max_uniform_quantize'>)


In [63]:
from ai_edge_quantizer import qtyping
import inspect

print(inspect.signature(qtyping.OpQuantizationConfig))

(activation_tensor_config: Optional[ai_edge_quantizer.qtyping.TensorQuantizationConfig] = None, weight_tensor_config: Optional[ai_edge_quantizer.qtyping.TensorQuantizationConfig] = None, compute_precision: ai_edge_quantizer.qtyping.ComputePrecision = <ComputePrecision.FLOAT: 'FLOAT'>, explicit_dequantize: bool = False, skip_checks: bool = False, min_weight_elements: int = 0) -> None


In [64]:
import inspect
from ai_edge_quantizer import qtyping

print(inspect.signature(qtyping.TensorQuantizationConfig))

(num_bits: int, symmetric: bool = True, granularity: ai_edge_quantizer.qtyping.QuantGranularity = <QuantGranularity.TENSORWISE: 'TENSORWISE'>, dtype: ai_edge_quantizer.qtyping.TensorDataType = <TensorDataType.INT: 'INT'>, algorithm_params: Mapping[str, Any] = <factory>) -> None


In [66]:
rep_ds = MVTecTrain(Path(CFG['data_root'])/'bottle', IMG)
rep_images = [rep_ds[i] for i in range(100)]
p, kb, ckpt = export_int8(STAGE2_DIR/'model_bottle_Baseline_FP32.pt', 'test_v2', rep_images)
print(f'INT8 size: {kb:.1f} KB  (was 1146 KB with the old converter; want ~452)')

# check dtypes
from ai_edge_litert.interpreter import Interpreter
it = Interpreter(model_path=str(p)); it.allocate_tensors()
dt = {}
for d in it.get_tensor_details():
    k = str(d['dtype']); dt[k] = dt.get(k,0)+1
print('dtypes:', dt)

(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:00) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:00)

...

(00:02) [ DONE] LiteRT-Torch Convert (+00:02)

(00:00) [START] Write Model to /content/stage3_out/tflite/test_v2_fp32.tflite

(00:00) [ DONE] Write Model to /content/stage3_out/tflite/test_v2_fp32.tflite (+00:00)

Skipping TFLOperationName.INPUT: Unsupported op for ComputePrecision.INTEGER: TFLOperationName.INPUT. Error: Quantization config for op: TFLOperationName.INPUT with config: OpQuantizationConfig(activation_tensor_config=None, weight_tensor_config=TensorQuantizationConfig(num_bits=8, symmetric=True, granularity=<QuantGranularity.CHANNELWISE: 'CHANNELWISE'>, dtype=<TensorDataType.INT: 'INT'>, algorithm_params=immutabledict({})), compute_precision=<ComputePrecision.INTEGER: 'INTEGER'>, explicit_dequantize=False, skip_checks=False, min_weight_elements=0) was not found in the policy.
Skipping TFLOperationName.OUTPUT: Unsupported op for ComputePrecision.INTEGER: TFLOperationName.OUTPUT. Error: Quantization config for op: TFLOperationName.OUTPUT with config: OpQuantizationConfig(activation_tensor_config=None, weight_tensor_config=TensorQuantizationConfig(num_bits=8, symmetric=True, granularity=<QuantGranularity.CHANNELWISE: 'CHANNELWISE'>, dtype=<TensorDataType.INT: 'INT'>, algorithm_params=im

/usr/local/lib/python3.12/dist-packages/ai_edge_litert/interpreter.py:472: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(


In [67]:
from ai_edge_litert.interpreter import Interpreter

it = Interpreter(model_path=str(p))
it.allocate_tensors()

dt = {}
for d in it.get_tensor_details():
    k = str(d['dtype'])
    dt[k] = dt.get(k, 0) + 1

print(dt)

{"<class 'numpy.float32'>": 20, "<class 'numpy.int32'>": 6, "<class 'numpy.int8'>": 8}


## 6. Verify: TFLite output ≈ PyTorch output
Critical step. A model that converts but produces wrong numbers is worse than one that fails loudly. We compare anomaly scores from the original PyTorch model vs the TFLite model on the test set, and check the AUROC is preserved.

In [45]:
import numpy as np

@torch.no_grad()
def score_pytorch(model, test_ds):
    model.eval(); b = PROTOCOL['border']; k = PROTOCOL['blur_kernel']
    scores, labels = [], []
    for x, y, d in DataLoader(test_ds, batch_size=16):
        err = (x - model(x)).pow(2).mean(1)
        err = err[:, b:-b, b:-b]
        err = F.avg_pool2d(err.unsqueeze(1), k, 1, k//2).squeeze(1)
        scores += err.flatten(1).mean(1).tolist(); labels += y.tolist()
    return np.array(scores), np.array(labels)

def score_litert(tflite_path, test_ds):
    """Score using LiteRT interpreter. litert_torch keeps NCHW layout."""
    from ai_edge_litert.interpreter import Interpreter
    interp = Interpreter(model_path=str(tflite_path))
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    in_dtype = inp['dtype']
    in_scale, in_zp = inp['quantization']
    out_scale, out_zp = out['quantization']

    b = PROTOCOL['border']; k = PROTOCOL['blur_kernel']
    scores, labels = [], []
    for x, y, d in test_ds:
        xn = x.unsqueeze(0).numpy().astype(np.float32)   # NCHW (1,3,128,128) — no permute!

        if in_dtype == np.int8:
            xq = np.round(xn / in_scale + in_zp).clip(-128, 127).astype(np.int8)
            interp.set_tensor(inp['index'], xq)
        else:
            interp.set_tensor(inp['index'], xn)

        interp.invoke()
        yq = interp.get_tensor(out['index'])[0].astype(np.float32)  # (3,128,128) CHW

        if out['dtype'] == np.int8:
            yhat = (yq - out_zp) * out_scale
        else:
            yhat = yq

        yhat = torch.tensor(yhat)                         # already CHW, no permute
        err = (x - yhat).pow(2).mean(0)[b:-b, b:-b]
        err = F.avg_pool2d(err[None, None], k, 1, k//2)[0, 0]
        scores.append(err.mean().item()); labels.append(y)
    return np.array(scores), np.array(labels)

test_ds = MVTecTest(Path(CFG['data_root'])/'bottle', IMG)
model, _ = rebuild_from_checkpoint(ckpt_file)

s_pt, l_pt = score_pytorch(model, test_ds)
s_tf, l_tf = score_litert(tflite_path, test_ds)        # ← just the path

print(f'PyTorch AUROC : {roc_auc_score(l_pt, s_pt):.4f}')
print(f'LiteRT  AUROC : {roc_auc_score(l_tf, s_tf):.4f}')
print(f'Score corr    : {np.corrcoef(s_pt, s_tf)[0,1]:.4f}  (want > 0.95)')

PyTorch AUROC : 0.9040
LiteRT  AUROC : 0.9040
Score corr    : 1.0000  (want > 0.95)


In [42]:
model, ckpt = rebuild_from_checkpoint(ckpt_file)

# Did the weights actually load? Compare against the checkpoint's stored AUROC.
print('Checkpoint says AUROC should be:', ckpt['auroc'])

# Re-score and check
s_pt, l_pt = score_pytorch(model, test_ds)
print('Rebuilt PyTorch AUROC:', roc_auc_score(l_pt, s_pt))

# Spot-check: are the weights non-trivial?
first_conv_w = model.enc[0][0].weight.data
print('First conv weight std:', first_conv_w.std().item(), '(should be ~0.05-0.2, not ~0)')

Checkpoint says AUROC should be: 0.835
Rebuilt PyTorch AUROC: 0.3976190476190476
First conv weight std: 0.0800536572933197 (should be ~0.05-0.2, not ~0)


In [43]:
print('TEST_NAME    :', TEST_NAME)
print('ckpt_file    :', ckpt_file.name)
print('ckpt category:', ckpt.get('category', '???'))
print('ckpt AUROC   :', ckpt['auroc'])
print('test_ds from :', 'bottle')   # whatever you set
print('tflite_path  :', tflite_path.name)

TEST_NAME    : model_bottle_Prune_50pct_l1
ckpt_file    : model_hazelnut_Prune_50pct_taylor.pt
ckpt category: hazelnut
ckpt AUROC   : 0.835
test_ds from : bottle
tflite_path  : model_bottle_Prune_50pct_l1.tflite


## 7. TFLite → C header for Arduino
`xxd`-style conversion: the .tflite bytes become a `const unsigned char[]` the firmware compiles in.

In [46]:
def tflite_to_header(tflite_path, var_name, out_dir=OUT/'headers'):
    data = Path(tflite_path).read_bytes()
    lines = [f'// Auto-generated from {Path(tflite_path).name}',
             f'// {len(data)} bytes', '#pragma once', '',
             f'const unsigned int {var_name}_len = {len(data)};',
             f'alignas(8) const unsigned char {var_name}[] = {{']
    for i in range(0, len(data), 12):
        chunk = ', '.join(f'0x{b:02x}' for b in data[i:i+12])
        lines.append(f'  {chunk},')
    lines.append('};')
    hpath = out_dir / f'{var_name}.h'
    hpath.write_text('\n'.join(lines))
    return hpath, len(data)

hpath, nbytes = tflite_to_header(tflite_path, 'model_bottle_prune50')
print(f'✓ Header: {hpath.name}  ({nbytes/1024:.1f} KB)')

✓ Header: model_bottle_prune50.h  (593.7 KB)


## 8. Batch-convert all deployment candidates
Run only after the single-model path above verifies. Converts each selected model, records real TFLite size, verifies AUROC, generates the header.

In [70]:
import numpy as np
import litert_torch

from ai_edge_quantizer import quantizer, qtyping
from ai_edge_litert.interpreter import Interpreter


def export_to_tflite_int8(ckpt_path, name, rep_images, out_dir=OUT):
    # ------------------------------------------------------------
    # 1. Rebuild model and export FP32 TFLite
    # ------------------------------------------------------------
    model, ckpt = rebuild_from_checkpoint(ckpt_path)
    model.eval()

    sample = (rep_images[0].unsqueeze(0),)

    fp32_path = out_dir / 'tflite' / f'{name}_fp32.tflite'

    litert_torch.convert(
        model.eval(),
        sample
    ).export(str(fp32_path))

    # ------------------------------------------------------------
    # 2. Create quantizer
    # ------------------------------------------------------------
    qt = quantizer.Quantizer(str(fp32_path))

    # Per-channel INT8 weights
    weight_cfg = qtyping.TensorQuantizationConfig(
        num_bits=8,
        symmetric=True,
        granularity=qtyping.QuantGranularity.CHANNELWISE
    )

    op_cfg = qtyping.OpQuantizationConfig(
        weight_tensor_config=weight_cfg,
        compute_precision=qtyping.ComputePrecision.INTEGER,
        explicit_dequantize=False
    )

    # Apply recipe to every supported TFLite op
    for op in qtyping.TFLOperationName:
        try:
            qt.update_quantization_recipe(
                regex='.*',
                operation_name=op,
                op_config=op_cfg,
                algorithm_key='min_max_uniform_quantize'
            )
        except Exception:
            pass

    # ------------------------------------------------------------
    # 3. Representative dataset for calibration
    # ------------------------------------------------------------
    interp = Interpreter(model_path=str(fp32_path))
    interp.allocate_tensors()

    input_name = interp.get_input_details()[0]['name']

    def representative_data():
        for img in rep_images[:100]:
            yield {
                input_name:
                    img.unsqueeze(0).cpu().numpy().astype(np.float32)
            }

    # ------------------------------------------------------------
    # 4. Calibrate + quantize
    # ------------------------------------------------------------
    calib = qt.calibrate(representative_data)
    result = qt.quantize(calib)

    int8_path = out_dir / 'tflite' / f'{name}.tflite'
    result.export_model(str(int8_path), overwrite=True)

    return int8_path, int8_path.stat().st_size / 1024, ckpt

In [71]:
DEPLOY = {
    'bottle': ['model_bottle_Baseline_FP32', 'model_bottle_Prune_50pct_l1', 'model_bottle_Distill_b16'],
    'hazelnut': ['model_hazelnut_Baseline_FP32', 'model_hazelnut_Distill_b16', 'model_hazelnut_Prune_50pct_taylor'],
}
results = []
for cat, names in DEPLOY.items():
    rep_ds = MVTecTrain(Path(CFG['data_root'])/cat, IMG)
    rep_images = [rep_ds[i] for i in range(min(100, len(rep_ds)))]
    test_ds = MVTecTest(Path(CFG['data_root'])/cat, IMG)
    for name in names:
        ckpt_file = STAGE2_DIR / f'{name}.pt'
        if not ckpt_file.exists():
            print(f'⚠️  missing {name}, skipping'); continue
        try:
            tfl, size_kb, ckpt = export_to_tflite_int8(ckpt_file, name + '_int8', rep_images)
            s_tf, l_tf = score_litert(tfl, test_ds)        # ← pass the PATH, no litert_torch.load
            auroc_tf = roc_auc_score(l_tf, s_tf)
            var = name.replace('model_', '')
            tflite_to_header(tfl, var)
            results.append({'category': cat, 'name': name, 'tflite_kb': round(size_kb,1),
                            'auroc_litert': round(auroc_tf,4),
                            'auroc_stage2': round(ckpt['auroc'],4), 'macs': ckpt['macs']})
            print(f'✓ {name}: {size_kb:.1f} KB, AUROC {auroc_tf:.4f} (Stage 2: {ckpt["auroc"]:.4f})')
        except Exception as e:
            print(f'✗ {name} FAILED: {str(e)[:300]}')

import pandas as pd
df = pd.DataFrame(results)
df.to_csv(OUT / 'tflite_results.csv', index=False)
print(); print(df.to_string())

(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:00) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:00)

...

(00:03) [ DONE] LiteRT-Torch Convert (+00:03)

(00:00) [START] Write Model to /content/stage3_out/tflite/model_hazelnut_Prune_50pct_taylor_int8_fp32.tflite

(00:00) [ DONE] Write Model to /content/stage3_out/tflite/model_hazelnut_Prune_50pct_taylor_int8_fp32.tflite 
(+00:00)

/usr/local/lib/python3.12/dist-packages/ai_edge_litert/interpreter.py:472: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(


Model name: /content/stage3_out/tflite/model_hazelnut_Prune_50pct_taylor_int8_fp32.tflite
Original model size: 593.69 KiB
Quantized model size: 160.32 KiB
Quantization Ratio: 0.27 (3.7x smaller)
Total time: 14.09 ms
✓ model_hazelnut_Prune_50pct_taylor: 160.3 KB, AUROC 0.8332 (Stage 2: 0.8350)

   category                               name  tflite_kb  auroc_litert  auroc_stage2       macs
0    bottle         model_bottle_Baseline_FP32      469.4        0.9302        0.9294  156620800
1    bottle        model_bottle_Prune_50pct_l1      160.3        0.9040        0.9040   44783616
2    bottle           model_bottle_Distill_b16      128.1        0.8159        0.8270   42683392
3  hazelnut       model_hazelnut_Baseline_FP32      469.4        0.8186        0.8150  156620800
4  hazelnut         model_hazelnut_Distill_b16      128.1        0.8550        0.8575   42683392
5  hazelnut  model_hazelnut_Prune_50pct_taylor      160.3        0.8332        0.8350   44783616


In [50]:
from ai_edge_litert.interpreter import Interpreter
interp = Interpreter(model_path=str(OUT/'tflite'/'model_bottle_Baseline_FP32_int8.tflite'))
interp.allocate_tensors()

dtypes = {}
for d in interp.get_tensor_details():
    dt = str(d['dtype'])
    dtypes[dt] = dtypes.get(dt, 0) + 1
print('Tensor dtypes in model:', dtypes)
print('Input dtype:', interp.get_input_details()[0]['dtype'])
print('Output dtype:', interp.get_output_details()[0]['dtype'])

Tensor dtypes in model: {"<class 'numpy.int8'>": 10, "<class 'numpy.float32'>": 15, "<class 'numpy.int32'>": 10}
Input dtype: <class 'numpy.int8'>
Output dtype: <class 'numpy.float32'>


## 9. Export a test image as C array (for on-device inference)
The Arduino needs one stored image to run inference on. We export a defect and a good image, INT8-quantized to match the model input.

In [72]:
def image_to_header(img_tensor, var_name, in_scale=1/255., in_zp=-128, out_dir=OUT/'headers'):
    x = img_tensor.permute(1,2,0).numpy()  # HWC float [0,1]
    xq = np.round(x/in_scale + in_zp).clip(-128,127).astype(np.int8).flatten()
    lines = [f'// test image {var_name}, INT8, {len(xq)} bytes (HWC {img_tensor.shape})',
             '#pragma once', f'const unsigned int {var_name}_len = {len(xq)};',
             f'const signed char {var_name}[] = {{']
    for i in range(0, len(xq), 12):
        lines.append('  ' + ', '.join(str(int(b)) for b in xq[i:i+12]) + ',')
    lines.append('};')
    p = out_dir / f'{var_name}.h'; p.write_text('\n'.join(lines)); return p

test_ds = MVTecTest(Path(CFG['data_root'])/'bottle', IMG)
good_i = next(i for i,(_,l,_) in enumerate(test_ds.items) if l==0)
bad_i  = next(i for i,(_,l,_) in enumerate(test_ds.items) if l==1)
image_to_header(test_ds[good_i][0], 'test_img_good')
image_to_header(test_ds[bad_i][0],  'test_img_defect')
print('✓ Exported test_img_good.h, test_img_defect.h')

✓ Exported test_img_good.h, test_img_defect.h


## 10. Package headers + back up to Drive

In [73]:
shutil.make_archive('/content/stage3_headers', 'zip', OUT/'headers')
shutil.copytree(OUT, '/content/drive/MyDrive/mvtec_stage3', dirs_exist_ok=True)
shutil.copy('/content/stage3_headers.zip', '/content/drive/MyDrive/mvtec_stage3/')
print('Headers + TFLite models backed up to Drive/mvtec_stage3')
print('Download stage3_headers.zip for the Arduino project.')
from google.colab import files
files.download('/content/stage3_headers.zip')

Headers + TFLite models backed up to Drive/mvtec_stage3
Download stage3_headers.zip for the Arduino project.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Zone 3 — On-Device (Half B)
*C++ firmware for the Arduino. These are reference cells — copy into the Arduino IDE, not run in Colab.*

## 11. Arduino setup

**Board:** Arduino Nano 33 BLE Sense Rev 2.  
**Library:** install `Arduino_TensorFlowLite` (or `Chirale_TensorFlowLite` / `Harvard_TinyMLx` depending on availability in 2026) via the Library Manager.

**Project files** (place in one sketch folder):
- `stage3_firmware.ino` — the firmware below
- `model_bottle_prune50.h` — a converted model header (start with one)
- `test_img_good.h`, `test_img_defect.h` — the test images

The model arena size (`kTensorArenaSize`) must be large enough to hold the model's activations. Start at 128 KB and shrink once you see the reported peak usage. The Nano 33 BLE has 256 KB SRAM total.

## 12. Firmware — inference + latency + memory

```cpp
#include <TensorFlowLite.h>
#include "tensorflow/lite/micro/all_ops_resolver.h"
#include "tensorflow/lite/micro/micro_interpreter.h"
#include "tensorflow/lite/schema/schema_generated.h"

#include "model_bottle_prune50.h"   // the model C array
#include "test_img_good.h"          // stored test image (INT8)
#include "test_img_defect.h"

// Tensor arena — holds model activations. Tune down after first run.
constexpr int kTensorArenaSize = 130 * 1024;
alignas(16) uint8_t tensor_arena[kTensorArenaSize];

const tflite::Model* model = nullptr;
tflite::MicroInterpreter* interpreter = nullptr;
TfLiteTensor* input = nullptr;
TfLiteTensor* output = nullptr;

// Compute mean reconstruction error (the anomaly score) over the image,
// matching the frozen scoring protocol (minus border crop / blur, which
// can be added on-device or applied host-side from the raw error).
float reconstruction_error(const signed char* in, const signed char* out,
                           float in_scale, int in_zp,
                           float out_scale, int out_zp, int n) {
  double acc = 0.0;
  for (int i = 0; i < n; i++) {
    float a = (in[i]  - in_zp)  * in_scale;
    float b = (out[i] - out_zp) * out_scale;
    acc += (double)(a - b) * (a - b);
  }
  return (float)(acc / n);
}

void setup() {
  Serial.begin(115200);
  while (!Serial);

  model = tflite::GetModel(model_bottle_prune50);
  if (model->version() != TFLITE_SCHEMA_VERSION) {
    Serial.println("Model schema mismatch!"); while (1);
  }

  static tflite::AllOpsResolver resolver;
  static tflite::MicroInterpreter static_interpreter(
      model, resolver, tensor_arena, kTensorArenaSize);
  interpreter = &static_interpreter;

  if (interpreter->AllocateTensors() != kTfLiteOk) {
    Serial.println("AllocateTensors failed — increase arena size"); while (1);
  }

  input  = interpreter->input(0);
  output = interpreter->output(0);

  Serial.print("Arena used (bytes): ");
  Serial.println(interpreter->arena_used_bytes());
  Serial.print("Input bytes: ");  Serial.println(input->bytes);
  Serial.print("Output bytes: "); Serial.println(output->bytes);
}

void run_one(const signed char* img, int img_len, const char* label) {
  // copy stored image into the input tensor
  for (int i = 0; i < img_len; i++) input->data.int8[i] = img[i];

  unsigned long t0 = micros();
  interpreter->Invoke();
  unsigned long t1 = micros();

  float score = reconstruction_error(
      input->data.int8, output->data.int8,
      input->params.scale, input->params.zero_point,
      output->params.scale, output->params.zero_point, img_len);

  Serial.print(label); Serial.print("  score="); Serial.print(score, 6);
  Serial.print("  latency_us="); Serial.println(t1 - t0);
}

void loop() {
  run_one(test_img_good,   test_img_good_len,   "GOOD  ");
  run_one(test_img_defect, test_img_defect_len, "DEFECT");
  delay(2000);
}
```

## 13. What to record on-device

For each deployed model, from the serial output and external instruments:

| Metric | Source |
|---|---|
| **Latency (µs)** | `micros()` around `Invoke()` — printed by firmware |
| **Arena / SRAM (bytes)** | `interpreter->arena_used_bytes()` — printed at setup |
| **Flash (bytes)** | model header size + sketch size (Arduino IDE compile output) |
| **Energy (mJ/inference)** | external: current shunt + scope, or Nordic PPK II. Measure current during `Invoke()`, integrate over latency, subtract idle baseline |
| **Score separation** | GOOD vs DEFECT scores — confirms the model works on-device |

Average latency/energy over many inferences (e.g., 100 loops) and report mean ± std.

## 14. Energy measurement note

USB power gives no energy variation — fine for measuring *per-inference cost*, which is what Stage 3 needs. The *adaptive* energy behaviour is Stage 4. For per-inference energy:

1. Power the board through a known shunt resistor (e.g. 1 Ω) in the supply line.
2. Measure voltage across the shunt on a scope during `Invoke()`.
3. Current = V_shunt / R; power = current × supply voltage; energy = power × latency.
4. Subtract the idle (between-inference) baseline to isolate inference energy.

Or use a Nordic Power Profiler Kit II for a turnkey current trace.

# Stage 3 — Report & Stage 4 Handoff

## What Stage 3 produces

- TFLite INT8 models for each deployment candidate, with **verified** AUROC (TFLite vs PyTorch) and **real** model size (replaces Stage 2's theoretical estimate).
- On-device measurements: latency, SRAM (arena), flash, and energy per inference.
- The Stage 2 Pareto plot, re-drawn with **measured** size/energy on the cost axis instead of estimated.

## Caveats

- **Conversion can change AUROC slightly.** Full INT8 (activations too, not just weights as in Stage 2's simulation) may shift AUROC up or down by 1–2 points. The verified TFLite AUROC is the real number; update the report.
- **Arena size is model-specific.** Report the measured `arena_used_bytes` per model — it's part of the deployability story (must fit 256 KB SRAM).
- **Latency may not track MACs linearly.** Memory access and layer types matter. If a low-MACs model is slow, that's a finding.

## Stage 4 — handoff

Stage 4 (energy-aware adaptation) needs:

1. **The measured energy per inference** for each model — this defines the cost of each rung on the quality ladder.
2. **The accuracy of each model** (verified TFLite AUROC) — the benefit of each rung.
3. **All selected models co-resident in flash** — the firmware must hold several models and switch between them at runtime.

The Stage 4 adaptive policy will pick, per inference, which model to run based on remaining energy budget, trading accuracy for energy as the (simulated) battery depletes.

### Open questions for Stage 4
- How many models can co-reside in 1 MB flash simultaneously? (Sum of header sizes.)
- What energy-budget thresholds map to which model? (Needs the measured per-inference energies.)
- Threshold policy vs utility policy — implement both, compare.